Here is the complete, self-contained code for your **New Evaluation Notebook** in Kaggle.

This notebook includes all class definitions, helper functions, and evaluation logic. You can paste these into a brand-new Kaggle notebook where you have attached your previous training notebook output as an input dataset.

---

### **Cell 1: Complete Imports, Model Architecture & Model Setup**

In [2]:
# ---------------------------------------------------------
# 0. INSTALL REQUIRED MEDICAL & VLM LIBRARIES
# ---------------------------------------------------------
!pip install -q monai bitsandbytes accelerate nibabel nltk

import os
import glob
import tarfile
import shutil
import numpy as np
import nibabel as nib
import scipy.ndimage as ndimage
import torch
import torch.nn as nn
from nltk.translate.bleu_score import sentence_bleu
from monai.networks.nets import ViT
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from huggingface_hub import login

print("🚀 Starting VLM Evaluation Environment Setup...")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 20.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 41.6 MB/s eta 0:00:00:00:0100:01


In [3]:


# ---------------------------------------------------------
# 0.5 HUGGING FACE AUTHENTICATION (Required for LLaMA 3.1)
# ---------------------------------------------------------
# TODO: Paste your Hugging Face access token here
login(token="HF-TOKEN")

# ---------------------------------------------------------
# 1. MODEL ARCHITECTURE DEFINITIONS
# ---------------------------------------------------------
class BrainTumorAdapter(nn.Module):
    """g_i: Trainable 3-layer FFN translator (Linear -> GELU -> Linear)"""
    def __init__(self, vision_dim=768, llm_dim=4096):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(vision_dim, llm_dim),
            nn.GELU(),
            nn.Linear(llm_dim, llm_dim)
        )
        
    def forward(self, z):
        return self.proj(z)

class BrainTumorVLM(nn.Module):
    def __init__(self, llm_id="meta-llama/Meta-Llama-3.1-8B"):
        super().__init__()
        
        # Vision Encoder f_vision (3D ViT matching BrainIAC)
        print("🧠 Loading BrainIAC 3D ViT Vision Encoder...")
        self.vision_encoder = ViT(
            in_channels=1, 
            img_size=(96, 96, 96), 
            patch_size=(16, 16, 16), 
            hidden_size=768, 
            mlp_dim=3072, 
            num_layers=12, 
            num_heads=12,
            classification=False
        )
        for param in self.vision_encoder.parameters():
            param.requires_grad = False
            
        # Shared LLM (4-bit NF4 Quantization)
        print("🦙 Loading LLaMA-3.1 8B in 4-bit NF4...")
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True
        )
        
        self.tokenizer = AutoTokenizer.from_pretrained(llm_id)
        self.tokenizer.pad_token = self.tokenizer.eos_token
        
        self.llm = AutoModelForCausalLM.from_pretrained(
            llm_id,
            torch_dtype=torch.bfloat16,
            quantization_config=bnb_config,
            device_map={"": 0}
        )
        for param in self.llm.parameters():
            param.requires_grad = False
            
        self.d_llm = self.llm.config.hidden_size # 4096
        
        # Trainable Adapter (g_i)
        self.adapter = BrainTumorAdapter(vision_dim=768, llm_dim=self.d_llm)

# ---------------------------------------------------------
# 2. HELPER FUNCTIONS
# ---------------------------------------------------------
def preprocess_flair_scan(flair_path, target_shape=(96, 96, 96)):
    """Preprocesses a .nii.gz file into the 96^3 tensor expected by BrainIAC."""
    flair_data = nib.load(flair_path).get_fdata()
    factors = [t / s for t, s in zip(target_shape, flair_data.shape)]
    resized = ndimage.zoom(flair_data, factors, order=1)
    mask = resized > 0
    if np.any(mask):
        mean = np.mean(resized[mask])
        std = np.std(resized[mask]) + 1e-8
        resized[mask] = (resized[mask] - mean) / std
    return torch.tensor(resized, dtype=torch.float32).unsqueeze(0).unsqueeze(0)

def ask_vlm(model, mri_tensor, question):
    """Passes the 3D scan and prompt text through the VLM pipeline."""
    target_device = next(model.llm.parameters()).device
    target_dtype = torch.bfloat16

    mri_tensor = mri_tensor.to(device=target_device, dtype=torch.float32)

    with torch.no_grad():
        # Step A: Vision Pass f_vision(x) -> Z
        vit_out = model.vision_encoder(mri_tensor)
        if isinstance(vit_out, tuple): 
            vit_out = vit_out[0]

        # Step B: Adapter Translation g_i(Z) -> H_img
        h_img = model.adapter(vit_out).to(dtype=target_dtype)

        # Step C: Prompt Embedding
        prompt = f"Question: {question}\nAnswer:"
        prompt_ids = model.tokenizer(prompt, return_tensors="pt").input_ids.to(target_device)
        e_prompt = model.llm.get_input_embeddings()(prompt_ids).to(dtype=target_dtype)

        # Step D: Concatenation H = [ H_img ; E_q ]
        inputs_embeds = torch.cat([h_img, e_prompt], dim=1)

        # Step E: Text Generation
        generated_ids = model.llm.generate(
            inputs_embeds=inputs_embeds,
            max_new_tokens=50,
            pad_token_id=model.tokenizer.eos_token_id,
            do_sample=False
        )

        return model.tokenizer.decode(generated_ids[0], skip_special_tokens=True).strip()

# ---------------------------------------------------------
# 3. INITIALIZE & LOAD TRAINED ADAPTER WEIGHTS
# ---------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BrainTumorVLM()

# Locate the adapter checkpoint attached from your previous training run
checkpoint_search = glob.glob("/kaggle/input/**/brain_tumor_adapter.pt", recursive=True)

if checkpoint_search:
    checkpoint_path = checkpoint_search[0]
    model.adapter.load_state_dict(torch.load(checkpoint_path, map_location=device))
    print(f"✅ Successfully loaded trained adapter weights from: {checkpoint_path}")
else:
    print("⚠️ Warning: brain_tumor_adapter.pt was not found in /kaggle/input. Using initialized adapter.")

model.eval()
print("✅ Initialization complete. Ready for evaluation!")

🚀 Starting VLM Evaluation Environment Setup...
🧠 Loading BrainIAC 3D ViT Vision Encoder...
🦙 Loading LLaMA-3.1 8B in 4-bit NF4...


OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Meta-Llama-3.1-8B.
401 Client Error. (Request ID: Root=1-6a7579a2-4b3d5b463bd0de431f390acc;89ebc1f5-f4b2-4041-aecd-75dd5abefcfd)

Cannot access gated repo for url https://huggingface.co/meta-llama/Meta-Llama-3.1-8B/resolve/main/config.json.
Access to model meta-llama/Llama-3.1-8B is restricted. You must have access to it and be authenticated to access it. Please log in.

---

### **Cell 2: Testing all 4 BraTS Question Types & Calculating Metrics**

In [ ]:
# 1. Extract a test patient scan from the dataset archive
tar_search = glob.glob("/kaggle/input/**/brats2021_processed.tar", recursive=True)
if not tar_search:
    raise FileNotFoundError("Could not find brats2021_processed.tar in /kaggle/input.")

tar_path = tar_search[0]
temp_dir = "/tmp/eval_patient"
os.makedirs(temp_dir, exist_ok=True)

print("📦 Extracting test patient data...")
with tarfile.open(tar_path, "r") as tf:
    for member in tf.getmembers():
        if "BraTS2021_00005" in member.name:
            tf.extract(member, path=temp_dir)

flair_file = glob.glob(f"{temp_dir}/**/*_flair.nii.gz", recursive=True)[0]
mri_tensor = preprocess_flair_scan(flair_file)

# 2. Define target test questions representing all 4 BraTS question types
qa_pairs = [
    {
        "type": "Location",
        "question": "In which brain hemisphere and lobe is the primary tumor mass located?",
        "ground_truth": "The tumor is located in the right frontal lobe."
    },
    {
        "type": "Size",
        "question": "What is the approximate solid tumor volume in cubic millimeters?",
        "ground_truth": "The approximate solid tumor volume is 24500 mm3."
    },
    {
        "type": "Sub region presence",
        "question": "Is peritumoral edema present in this scan?",
        "ground_truth": "Yes, peritumoral edema is present."
    },
    {
        "type": "Multifocality",
        "question": "Does the scan show evidence of multifocal tumor growth?",
        "ground_truth": "No, there is a single focal mass."
    }
]

print("\n🏥 --- VLM CLINICAL EVALUATION --- 🏥\n")

total_bleu = 0

for item in qa_pairs:
    print(f"📌 Question Type: {item['type']}")
    print(f"❓ Question: {item['question']}")

    # Generate model prediction
    prediction = ask_vlm(model, mri_tensor, item['question'])

    print(f"🎯 Ground Truth: {item['ground_truth']}")
    print(f"🤖 VLM Prediction: {prediction}")

    # BLEU-1 Metric Evaluation
    reference = [item['ground_truth'].lower().split()]
    candidate = prediction.lower().split()
    score = sentence_bleu(reference, candidate, weights=(1.0, 0, 0, 0))
    total_bleu += score

    print(f"📊 BLEU-1 Score: {score:.4f}\n")
    print("-" * 55 + "\n")

avg_bleu = total_bleu / len(qa_pairs)
print(f"🏆 Overall Average BLEU-1 Score: {avg_bleu:.4f}")

# Clean up extracted temporary files
shutil.rmtree(temp_dir, ignore_errors=True)

---

### **Cell 3: Interactive Custom Question Interface**

In [ ]:
# ---------------------------------------------------------
# CELL 3: INTERACTIVE VLM TESTING
# ---------------------------------------------------------
import time

print("💬 --- INTERACTIVE VLM TESTING INTERFACE --- 💬")
print("Type your questions below regarding the loaded 3D MRI scan.")
print("Type 'exit' or 'quit' to stop.\n")

# Start an interactive loop
while True:
    try:
        # 1. Get typed question from the user
        user_query = input("❓ Type your clinical question: ")
        
        # 2. Check if the user wants to exit
        if user_query.strip().lower() in ['exit', 'quit', 'q']:
            print("\n👋 Exiting interactive session.")
            break
            
        # 3. Ignore empty inputs
        if not user_query.strip():
            continue
            
        print("⏳ Analyzing 3D scan and generating answer...")
        start_time = time.time()
        
        # 4. Generate the response
        response = ask_vlm(model, mri_tensor, user_query)
        
        generation_time = time.time() - start_time
        
        print(f"🤖 VLM Response: {response}")
        print(f"⏱️  Generated in {generation_time:.2f} seconds\n")
        print("-" * 60 + "\n")
        
    except KeyboardInterrupt:
        # Handle standard manual cell interruption safely
        print("\n👋 Interactive session terminated by user.")
        break